# NISAR GCOV — Raster/QGIS Preparation


## Geographic checkpoint

Same workflow as before, just with the spatial context made explicit so the outputs stay traceable back to the scene and AOI.

# 10 — GeoTIFF Export & QGIS Validation

Open what Module 09 exported in QGIS and confirm the CRS and extent match what you recorded back in Module 01.

In [ ]:
from pathlib import Path
from nisar_utils.bootstrap import setup_workshop
WORKSHOP_ROOT = setup_workshop()

from nisar_utils.config import load_config
from nisar_utils.workflow import (
    build_profile, resolve_frequency, resolve_terms,
    resolve_aoi, resolve_window
)

cfg = load_config()
NISAR_FILE = Path(cfg["nisar_file"])
profile = build_profile(cfg)
freq = resolve_frequency(cfg, profile)
terms, diagonal_terms, off_diagonal_terms = resolve_terms(profile, freq)

print("File:", NISAR_FILE)
print("SAR family:", profile.sar_family)
print("Band:", profile.band)
print("Level:", profile.product_level)
print("Product:", profile.product_type)
print("GCOV root:", profile.gcov_root)
print("Frequency:", freq)
print("Polarization:", profile.polarization_channels)


In [ ]:
from nisar_utils.gcov import open_gcov, get_grid_coordinates, get_projection_info, read_window
from nisar_utils.spatial import coordinate_to_index

grid=f"{profile.gcov_root}/grids/{freq}"
term=(diagonal_terms or terms)[0]
with open_gcov(NISAR_FILE) as f:
    x,y=get_grid_coordinates(f,grid)
    projection=get_projection_info(f,grid)
    (r0,r1,c0,c1),_=resolve_window(x,y,cfg,coordinate_to_index_func=coordinate_to_index)
    arr=read_window(f,f"{grid}/{term}",r0,r1,c0,c1)

print("Export source:",f"{grid}/{term}")
print("Array:",arr.shape,arr.dtype)
print("EPSG:",profile.epsg)
print("Projection:",projection)
print("Window:",(r0,r1,c0,c1))
print("X spacing:",float(x[1]-x[0]) if len(x)>1 else "n/a")
print("Y spacing:",float(y[1]-y[0]) if len(y)>1 else "n/a")
print("\nModule 10 STATUS: PASS")


In [ ]:
from IPython.display import display

try:
    from nisar_utils.gcov import open_gcov, get_grid_coordinates
    from nisar_utils.mapping import folium_scene_map

    _grid = f"{profile.gcov_root}/grids/{freq}"

    with open_gcov(NISAR_FILE) as _f:
        _x, _y = get_grid_coordinates(_f, _grid)

    print("Grid:", _grid)
    print("X coordinates:", len(_x))
    print("Y coordinates:", len(_y))
    print("EPSG:", profile.epsg)

    # ------------------------------------------------------------
    # Normalize the geographic AOI saved by Module 06
    # ------------------------------------------------------------
    _aoi10 = cfg.get("default_aoi") or cfg.get("aoi")

    print("Stored AOI:", _aoi10)

    if _aoi10 and all(k in _aoi10 for k in
                      ("xmin", "xmax", "ymin", "ymax")):

        _map_aoi10 = {
            "lon_min": float(_aoi10["xmin"]),
            "lon_max": float(_aoi10["xmax"]),
            "lat_min": float(_aoi10["ymin"]),
            "lat_max": float(_aoi10["ymax"]),
        }

    elif _aoi10 and all(k in _aoi10 for k in
                        ("lon_min", "lon_max", "lat_min", "lat_max")):

        _map_aoi10 = {
            "lon_min": float(_aoi10["lon_min"]),
            "lon_max": float(_aoi10["lon_max"]),
            "lat_min": float(_aoi10["lat_min"]),
            "lat_max": float(_aoi10["lat_max"]),
        }

    else:
        _map_aoi10 = None

    print("Mapping AOI:", _map_aoi10)

    # ------------------------------------------------------------
    # Create and explicitly display the GIS checkpoint map
    # ------------------------------------------------------------
    _m = folium_scene_map(
        _x,
        _y,
        profile.epsg,
        title="NISAR GIS checkpoint",
        aoi=_map_aoi10
    )

    print("Map object created:", type(_m))

    display(_m)

except Exception as e:
    print("Spatial checkpoint unavailable:")
    print(type(e).__name__, ":", e)
